# Demo of the trained model

In [1]:
import numpy as np

from omegaconf import OmegaConf
from word2vec.evaluation.benchmark import run_benchmark
from word2vec.evaluation.knn import knn_demo
from word2vec.model import load_model_for_config
from word2vec.evaluation.arithmetic import test_arithmetic_operations

/home/maksym/projects/word2vec/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cfg = OmegaConf.load("conf/config.yaml")
print(OmegaConf.to_yaml(cfg))

dataset: text8
stage: preprocess
preprocessing:
  neg_sampling_dist_exponent: 0.75
  min_token_corpus_count: 5
  unigram_table_size: 100000000
  force_preprocess: false
  subsampling_threshold: 1.0e-05
training:
  max_neighbourhood_size: 5
  latent_dimensionality: 300
  num_negative_samples: 10
  lr_start: 0.025
  num_epochs: 5
  seed: 42
  force_train: false
benchmark:
  dataset: simlex999
knn:
  words:
  - pasta
  - four
  - ronald
  - fruit
  - guitar
  - happiness
arithmetic:
  patterns:
  - king,man,woman,queen
  - paris,france,italy,rome
  - walking,walk,swim,swimming
  - biggest,big,small,smallest
  - cat,kitten,puppy,dog
  - banana,yellow,red,apple
  - teacher,school,hospital,doctor



In [3]:
model = load_model_for_config(cfg)

2026-03-10 00:18:39.368 | INFO     | word2vec.model:load_model_for_config:21 - Model found at /home/maksym/projects/word2vec/models/text8_d300_k10_e5.pkl, loading from disk


In [4]:
vocab_size, dim = model.embeddings.shape

print("Model Summary")
print("----------------------")
print(f"Vocabulary size:        {vocab_size:,}")
print(f"Embedding dimension:    {dim}")
print(f"Embedding dtype:        {model.embeddings.dtype}")

norms = np.linalg.norm(model.embeddings, axis=1)
print("\nEmbedding norm statistics")
print("-------------------------")
print(f"Mean norm:               {norms.mean():.4f}")
print(f"Std norm:                {norms.std():.4f}")
print(f"Min norm:                {norms.min():.4f}")
print(f"Max norm:                {norms.max():.4f}")

variances = np.var(model.embeddings, axis=0)
print("\nPer-dimension variance")
print("----------------------")
print(f"Mean variance:           {variances.mean():.6f}")
print(f"Min variance:            {variances.min():.6f}")
print(f"Max variance:            {variances.max():.6f}")

rng = np.random.default_rng(0)
i = rng.integers(0, vocab_size, 10000)
j = rng.integers(0, vocab_size, 10000)

cosines = np.sum(model.emb_norm[i] * model.emb_norm[j], axis=1)
print("\nRandom cosine similarity check")
print("-------------------------------")
print(f"Mean cosine:    {cosines.mean():.4f}")
print(f"Std cosine:     {cosines.std():.4f}")

Model Summary
----------------------
Vocabulary size:        71,290
Embedding dimension:    300
Embedding dtype:        float32

Embedding norm statistics
-------------------------
Mean norm:               2.4777
Std norm:                0.5953
Min norm:                1.5170
Max norm:                5.5527

Per-dimension variance
----------------------
Mean variance:           0.011085
Min variance:            0.005922
Max variance:            0.019534

Random cosine similarity check
-------------------------------
Mean cosine:    0.5513
Std cosine:     0.1403


We can see that there is some anisotropy going on in the embeddings - the mean cosine sim is pretty high.

In [5]:
knn_demo(cfg)

2026-03-10 00:18:39.624 | INFO     | word2vec.model:load_model_for_config:21 - Model found at /home/maksym/projects/word2vec/models/text8_d300_k10_e5.pkl, loading from disk



Top 5 nearest neighbors for pasta:
1. savoury: 0.8694
2. sausage: 0.8567
3. desserts: 0.8557
4. veal: 0.8530
5. liqueur: 0.8499

Top 5 nearest neighbors for four:
1. three: 0.9424
2. six: 0.9397
3. five: 0.9372
4. seven: 0.9277
5. eight: 0.9147

Top 5 nearest neighbors for ronald:
1. reagan: 0.7949
2. seretse: 0.7514
3. mondale: 0.7510
4. mcnamara: 0.7500
5. agnew: 0.7500

Top 5 nearest neighbors for fruit:
1. fruits: 0.8454
2. vegetables: 0.8264
3. juices: 0.8150
4. vegetable: 0.7964
5. rotting: 0.7927

Top 5 nearest neighbors for guitar:
1. bass: 0.8140
2. guitars: 0.7853
3. harmonica: 0.7806
4. saxophone: 0.7734
5. clavichord: 0.7625

Top 5 nearest neighbors for happiness:
1. goodness: 0.7697
2. toil: 0.7658
3. kindness: 0.7630
4. infirmities: 0.7599
5. unhappiness: 0.7582


In [6]:
run_benchmark(cfg)

2026-03-10 00:18:39.741 | INFO     | word2vec.model:load_model_for_config:21 - Model found at /home/maksym/projects/word2vec/models/text8_d300_k10_e5.pkl, loading from disk
2026-03-10 00:18:42.127 | INFO     | word2vec.evaluation.benchmark:eval_simlex:63 - Simlex999 benchmark results: coverage=0.993, corr=0.2985


In [7]:
test_arithmetic_operations(cfg)

2026-03-10 00:18:42.142 | INFO     | word2vec.model:load_model_for_config:21 - Model found at /home/maksym/projects/word2vec/models/text8_d300_k10_e5.pkl, loading from disk



Pattern: king - man + woman = ? (Expected: queen)
  king: 0.8148
  montferrat: 0.6856
  infanta: 0.6845
  zadok: 0.6816
  bessus: 0.6805

Pattern: paris - france + italy = ? (Expected: rome)
  paris: 0.8316
  bologna: 0.7125
  italy: 0.7099
  vevey: 0.6786
  cassino: 0.6760

Pattern: walking - walk + swim = ? (Expected: swimming)
  swim: 0.7103
  walking: 0.6807
  coyotes: 0.6030
  biking: 0.6011
  hiking: 0.5916

Pattern: biggest - big + small = ? (Expected: smallest)
  small: 0.7250
  biggest: 0.7194
  largest: 0.6700
  large: 0.6255
  poorest: 0.6005

Pattern: cat - kitten + puppy = ? (Expected: dog)
  cat: 0.9171
  puppy: 0.7461
  eared: 0.7281
  guanaco: 0.7218
  rattus: 0.7042

Pattern: banana - yellow + red = ? (Expected: apple)
  banana: 0.8308
  red: 0.6301
  daiquiri: 0.6059
  iced: 0.5994
  buna: 0.5969

Pattern: teacher - school + hospital = ? (Expected: doctor)
  hospital: 0.7443
  teacher: 0.7136
  nurse: 0.6606
  housekeeper: 0.6191
  schoolteacher: 0.6124
